In [ ]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

In [ ]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [ ]:
# add your code here - consider creating a new cell for each section of code

# 1) Remove users with < 200 ratings and books with < 100 ratings
#    (counts are computed on the full, unfiltered ratings data)
user_counts = df_ratings['user'].value_counts()
book_counts = df_ratings['isbn'].value_counts()

df = df_ratings[~df_ratings['user'].isin(user_counts[user_counts < 200].index)]
df = df[~df['isbn'].isin(book_counts[book_counts < 100].index)]

print('Ratings after filtering:', df.shape)

In [ ]:
# 2) Attach titles, drop duplicate (title, user) pairs, build title x user matrix
df = pd.merge(df, df_books, on='isbn')
df = df.drop_duplicates(['title', 'user'])

book_user_matrix = df.pivot(index='title', columns='user', values='rating').fillna(0)
book_user_sparse = csr_matrix(book_user_matrix.values)

print('Matrix shape (books x users):', book_user_matrix.shape)

In [ ]:
# 3) Fit the KNN model using cosine distance
model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(book_user_sparse)

In [ ]:
# function to return recommended books - this will be tested
def get_recommends(book = ""):
  # vector of ratings for the requested book
  book_vector = book_user_matrix.loc[[book]].values

  # 6 neighbours: the first one is the book itself (distance 0)
  distances, indices = model.kneighbors(book_vector, n_neighbors=6)

  recs = []
  for dist, idx in zip(distances[0][1:], indices[0][1:]):
    recs.append([book_user_matrix.index[idx], float(dist)])

  # expected output lists the farthest of the 5 first
  recs.reverse()

  recommended_books = [book, recs]
  return recommended_books

In [ ]:
print(get_recommends("The Queen of the Damned (Vampire Chronicles (Paperback))"))

In [ ]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2): 
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()